In [0]:
# Inject WHL to bypass Serverless environment dependencies crash
%pip install /Workspace/Users/xh.shi0507@gmail.com/.bundle/enterprise-market-macro-lakehouse/artifacts/.internal/lakehouse-0.1.0-py3-none-any.whl -q
dbutils.library.restartPython()

# 0.Ingest ECB euro foreign exchange reference rates to Bronze (raw XML as string)
   Target: enterprise.bronze_ref.ecb_fx_raw

# 1. Imports & Dependencies

In [0]:
import uuid
from datetime import datetime, timezone

import requests
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql import types as T

# 2.Widget Definitions

In [0]:
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("schema", "bronze_ref")
dbutils.widgets.text("table", "ecb_fx_raw")

dbutils.widgets.text("base_ccy", "EUR")
# dbutils.widgets.dropdown("mode", "daily", ["daily", "hist_90d"])  # daily or last 90 days

dbutils.widgets.dropdown("mode", "hist_90d", ["daily", "hist_90d"])  # daily or last 90 days

dbutils.widgets.text("source", "ecb")  # optional, for lineage (if you want add later)

# 3. Configuration & Setup URL

In [0]:
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
TABLE = dbutils.widgets.get("table").strip()
BASE_CCY = dbutils.widgets.get("base_ccy").strip().upper()
MODE = dbutils.widgets.get("mode").strip()

TARGET_TBL = f"{CATALOG}.{SCHEMA}.{TABLE}"

URL_DAILY = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-daily.xml"
URL_90D = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist-90d.xml"

url = URL_DAILY if MODE == "daily" else URL_90D

run_id = str(uuid.uuid4())
ingestion_ts = datetime.now(timezone.utc)

print(f"[INFO] target={TARGET_TBL}")
print(f"[INFO] base_ccy={BASE_CCY}, mode={MODE}, url={url}")
print(f"[INFO] run_id={run_id}, ingestion_ts={ingestion_ts.isoformat()}")

# 4. DDL / Table `Initialization`

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TBL} (
  base_ccy      STRING,
  raw_json      STRING,
  run_id        STRING,
  ingestion_ts  TIMESTAMP
)
USING DELTA
""")

# 5. Data Extraction

In [0]:
r = requests.get(url, timeout=30)
r.raise_for_status()
raw_xml = r.text

# 6. Data Writing

In [0]:
row = Row(base_ccy=BASE_CCY, raw_json=raw_xml, run_id=run_id, ingestion_ts=ingestion_ts)

schema = T.StructType(
    [
        T.StructField("base_ccy", T.StringType(), False),
        T.StructField("raw_json", T.StringType(), False),
        T.StructField("run_id", T.StringType(), False),
        T.StructField("ingestion_ts", T.TimestampType(), False),
    ]
)

df = spark.createDataFrame([row], schema=schema)

(df.write.mode("append").format("delta").saveAsTable(TARGET_TBL))

print("[INFO] ECB bronze write completed.")

display(spark.table(TARGET_TBL).where(F.col("run_id") == run_id))